# OCR-Free Model Comparison

## Setup

In [1]:
# Import Libraries

import ollama
import json
import re
import time
from rapidfuzz import fuzz

### Config

In [2]:
# Models
MODELS = ["qwen2.5vl:3b", "gemma4:e2b"]

### Test Cases

In [3]:
# Test Cases
TEST_CASES = [
    {
        "image": "../data/receipt/bon-solaria.jpeg",
        "ground_truth": {
            "items": [
                {
                    "name": "Siomay",
                    "quantity": 2,
                    "price": 20002
                },
                {
                    "name": "Air Mineral Botol",
                    "quantity": 1,
                    "price": 10001
                },
                {
                    "name": "Es Teh Manis",
                    "quantity": 1,
                    "price": 8183 
                },
                {
                    "name": "Kwetiau Sapi Siram",
                    "quantity": 1,
                    "price": 44546
                },
                {
                    "name": "Kwetiau Seafood Goreng",
                    "quantity": 1,
                    "price": 46364
                }
            ],
            "charges": [
                {
                    "name": "PB1 10%",
                    "amount": 12910
                },
                {
                    "name": "Rounding",
                    "amount": -6
                }
            ],
            "currency": "IDR"
        }
    },
    {
        "image": "../data/receipt/bon-paul-bakery.jpg",
        "ground_truth": {
            "items": [
                {
                    "name": "Cappucino",
                    "quantity": 1,
                    "price": 45000
                },
                {
                    "name": "Cappucino",
                    "quantity": 1,
                    "price": 45000
                },
                {
                    "name": "Hazelnut Cappucino",
                    "quantity": 1,
                    "price": 50000 
                },
                {
                    "name": "Grn Apple Celery Jui",
                    "quantity": 1,
                    "price": 60000
                },
                {
                    "name": "Croissant Cheese",
                    "quantity": 1,
                    "price": 36000
                },
                {
                    "name": "Hazelnut Cappucino",
                    "quantity": 1,
                    "price": 50000 
                }
            ],
            "charges": [
                {
                    "name": "Service",
                    "amount": 14300
                },
                {
                    "name": "PB1",
                    "amount": 30030
                }
            ],
            "currency": "IDR"
        }
    },
    {
        "image": "../data/receipt/bon-jco.jpg",
        "ground_truth": {
            "items": [
                {
                    "name": "JPops 2 Dzn A",
                    "quantity": 1,
                    "price": 48000
                }
            ],
            "charges": [],
            "currency": "IDR"
        }
    },
    {
        "image": "../data/receipt/bon-hokben.jpg",
        "ground_truth": {
            "items": [
                {
                    "name": "Beef Teriyaki RB",
                    "quantity": 2,
                    "price": 65456
                },
                {
                    "name": "Rice Only RB",
                    "quantity": 2,
                    "price": 10910
                },
                {
                    "name": "Ogura RB",
                    "quantity": 1,
                    "price": 9091
                },
                {
                    "name": "Cold Ocha",
                    "quantity": 2,
                    "price": 10910
                }
            ],
            "charges": [
                {
                    "name": "PJK Resto 10%",
                    "amount": 9637
                },
                {
                    "name": "Pembulatan",
                    "amount": 4
                }
            ],
            "currency": "IDR"
        }
    },
]

| # | Receipt | Unique Challenge |
|---|---|---|
| 1 | Solaria | Two-line item, negative rounding |
| 2 | Paul Bakery | Duplicate entries, no quantity |
| 3 | JCO | Package with sub-items, PB1 included |
| 4 | HokBen | Blurry image, Pembulatan |

## Benchmarking

### Create Benchmarking Building Blocks

In [4]:
SYSTEM_PROMPT = """
You are a receipt parser. Extract only the relevant information from the receipt image.

ITEMS:
- An item is anything that was ordered (food, drinks, or services) with a price
- If the same item appears multiple times as separate lines, list each one separately
- Include all items regardless of whether the name appears abbreviated or unusual — extract exactly as written on the receipt
- If a line has no price and is at the same indentation level as the previous item, consolidate it into the previous item's name
- If a line has no price and is indented beneath the previous item, it is a sub-item description — ignore it

CHARGES:
- A charge is any financial adjustment such as tax, service charge, or discount with an explicit amount listed next to it
- Any line with an explicit amount positioned between Sub Total and Total is a charge — include it even if it appears near summary lines
- Only include a charge if it has an explicit amount listed next to it — ignore any text that merely mentions a charge within a sentence

IGNORE:
- Summary lines: Sub Total, Total, Grand Total, Cash, Change, Taxable Amount, Taxable Amt, Before Rounding, Payment
- Restaurant name, address, taglines, cashier info, loyalty points, table numbers
- Any text without a clear standalone price

Return ONLY a valid JSON object with no explanation or markdown. Use this exact schema:
{
  "items": [
    {"name": "item name", "quantity": 1, "price": 0.00}
  ],
  "charges": [
    {"name": "charge name", "amount": 0.00}
  ],
  "currency": "IDR"
}

Example:
Input: A receipt with nasi goreng, es teh, and 11% tax
Output:
{
  "items": [
    {"name": "Nasi Goreng", "quantity": 1, "price": 45000.00},
    {"name": "Es Teh", "quantity": 2, "price": 16000.00}
  ],
  "charges": [
    {"name": "PPN 11%", "amount": 6710.00}
  ],
  "currency": "IDR"
}
"""

In [5]:
import base64
from io import BytesIO
from PIL import Image

def resize_image(image_path: str, max_size: int = 1024) -> str:
    """Resize image and return as base64 string."""
    img = Image.open(image_path)
    img.thumbnail((max_size, max_size))
    buffer = BytesIO()
    img.save(buffer, format=img.format or "JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

def clean_json_output(raw: str) -> dict:
    """Strip markdown fences, remove thousands separators, and parse JSON."""
    # Strip markdown fences
    cleaned = re.sub(r"```json|```", "", raw).strip()
    # Remove thousands separators in numbers e.g. 9,637 → 9637
    cleaned = re.sub(r'(\d),(\d{3})', r'\1\2', cleaned)
    return json.loads(cleaned)

def parse_receipt(image_input: str, model: str) -> dict:
    """Send receipt image to VLM and return parsed JSON."""
    try:
        image_input = resize_image(image_input)  # 👈 resize + convert to base64
        
        response = ollama.chat(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": "Here is the receipt.",
                    "images": [image_input]
                }
            ]
        )
        raw = response["message"]["content"]
        return clean_json_output(raw)

    except json.JSONDecodeError:
        return {"error": "Model returned invalid JSON", "raw": raw}
    except Exception as e:
        return {"error": str(e)}

In [6]:
def score_results(predicted: dict, ground_truth: dict) -> dict:
    """Compare predicted output against ground truth"""
    # --- Score items ---
    name_scores = []
    price_matches = []

    for exp in ground_truth["items"]:
        best_name_score = 0
        best_price_match = False

        for pred in predicted.get("items", []):
            name_score = fuzz.ratio(
                pred["name"].lower(),
                exp["name"].lower()
            ) / 100
            if name_score > best_name_score:
                best_name_score = name_score
                best_price_match = (pred["price"] == exp["price"])

        name_scores.append(best_name_score)
        price_matches.append(best_price_match)

    # --- Score charges ---
    charge_name_scores = []
    charge_amount_matches = []

    for exp in ground_truth["charges"]:
        best_name_score = 0
        best_amount_match = False

        for pred in predicted.get("charges", []):
            name_score = fuzz.ratio(
                pred["name"].lower(),
                exp["name"].lower()
            ) / 100
            if name_score > best_name_score:
                best_name_score = name_score
                best_amount_match = (pred["amount"] == exp["amount"])

        charge_name_scores.append(best_name_score)
        charge_amount_matches.append(best_amount_match)

    return {
        "avg_item_name_score": sum(name_scores) / len(name_scores) if name_scores else 0,
        "item_price_accuracy": sum(price_matches) / len(price_matches) if price_matches else 0,
        "avg_charge_name_score": sum(charge_name_scores) / len(charge_name_scores) if charge_name_scores else 1.0,
        "charge_amount_accuracy": sum(charge_amount_matches) / len(charge_amount_matches) if charge_amount_matches else 1.0,
    }

In [7]:
results = {"qwen2.5vl:3b": [], "gemma4:e2b": []}

### Run Benchmarking

In [8]:
# Warmup
print("Warming up models...")
for model in MODELS:
    ollama.chat(
        model=model,
        messages=[{"role": "user", "content": "hi"}]
    )
    print(f"✅ {model} ready")
print("Warmup complete!\n")

Warming up models...
✅ qwen2.5vl:3b ready
✅ gemma4:e2b ready
Warmup complete!



In [9]:
results = {"qwen2.5vl:3b": [], "gemma4:e2b": []}

for model in MODELS:
    for test in TEST_CASES:
        start = time.time()
        predicted = parse_receipt(test["image"], model)
        end = time.time()
        
        results[model].append({
            "receipt": test["image"].split("/")[-1],
            "time": end - start,
            "predicted": predicted,
            "ground_truth": test["ground_truth"]
        })

### Print Results for Qwen2.5VL

In [10]:
for run in results["qwen2.5vl:3b"]:
    print(f"\n{'='*50}")
    print(f"Model: qwen2.5vl:3b | Receipt: {run['receipt']}")
    
    if "error" in run["predicted"]:
        print(f"ERROR: {run['predicted']['error']}")
        print("="*50)
        continue
    
    print(f"\nPREDICTED:\n{json.dumps(run['predicted'], indent=2)}")
    print(f"\nGROUND TRUTH:\n{json.dumps(run['ground_truth'], indent=2)}")
    
    scores = score_results(run["predicted"], run["ground_truth"])
    print(f"\nSCORES:\n{json.dumps(scores, indent=2)}")
    print(f"Response time: {run['time']:.2f}s")
    print("="*50)


Model: qwen2.5vl:3b | Receipt: bon-solaria.jpeg

PREDICTED:
{
  "items": [
    {
      "name": "Siomay",
      "quantity": 2,
      "price": 20000.0
    },
    {
      "name": "Air Mineral Botol",
      "quantity": 1,
      "price": 10001.0
    },
    {
      "name": "Es Teh Manis",
      "quantity": 1,
      "price": 8183.0
    },
    {
      "name": "Kwetiau Sapi Siram",
      "quantity": 1,
      "price": 44546.0
    },
    {
      "name": "Kwetiau Seafood",
      "quantity": 1,
      "price": 46364.0
    },
    {
      "name": "Goreng",
      "quantity": 1,
      "price": 129096.0
    }
  ],
  "charges": [
    {
      "name": "Pb1 10%",
      "amount": 12910.0
    }
  ],
  "currency": "IDR"
}

GROUND TRUTH:
{
  "items": [
    {
      "name": "Siomay",
      "quantity": 2,
      "price": 20002
    },
    {
      "name": "Air Mineral Botol",
      "quantity": 1,
      "price": 10001
    },
    {
      "name": "Es Teh Manis",
      "quantity": 1,
      "price": 8183
    },
    {
    

### Print Results for Gemma4

In [11]:
for run in results["gemma4:e2b"]:
    print(f"\n{'='*50}")
    print(f"Model: gemma4:e2b | Receipt: {run['receipt']}")
    
    if "error" in run["predicted"]:
        print(f"ERROR: {run['predicted']['error']}")
        print("="*50)
        continue
    
    print(f"\nPREDICTED:\n{json.dumps(run['predicted'], indent=2)}")
    print(f"\nGROUND TRUTH:\n{json.dumps(run['ground_truth'], indent=2)}")
    
    scores = score_results(run["predicted"], run["ground_truth"])
    print(f"\nSCORES:\n{json.dumps(scores, indent=2)}")
    print(f"Response time: {run['time']:.2f}s")
    print("="*50)


Model: gemma4:e2b | Receipt: bon-solaria.jpeg

PREDICTED:
{
  "items": [
    {
      "name": "Siomay",
      "quantity": 1,
      "price": 20002.0
    },
    {
      "name": "Air Mineral Botol",
      "quantity": 1,
      "price": 10001.0
    },
    {
      "name": "Es Teh Manis",
      "quantity": 1,
      "price": 8183.0
    },
    {
      "name": "Kwetiau Sapi Siram",
      "quantity": 1,
      "price": 44546.0
    },
    {
      "name": "Kwetiau Seafood",
      "quantity": 1,
      "price": 46364.0
    }
  ],
  "charges": [
    {
      "name": "Pb1 10%",
      "amount": 0.0
    }
  ],
  "currency": "IDR"
}

GROUND TRUTH:
{
  "items": [
    {
      "name": "Siomay",
      "quantity": 2,
      "price": 20002
    },
    {
      "name": "Air Mineral Botol",
      "quantity": 1,
      "price": 10001
    },
    {
      "name": "Es Teh Manis",
      "quantity": 1,
      "price": 8183
    },
    {
      "name": "Kwetiau Sapi Siram",
      "quantity": 1,
      "price": 44546
    },
    {
 

### Summary

In [12]:
print(f"{'Model':<20} {'Receipt':<25} {'Time':>6} {'ItemName':>10} {'ItemPrice':>10} {'ChgName':>9} {'ChgAmt':>9}")
print("-" * 95)

for model, runs in results.items():
    for run in runs:
        if "error" in run["predicted"]:
            print(f"{model:<20} {run['receipt']:<25} {run['time']:>6.2f}s {'ERROR':>10}")
            continue
        
        scores = score_results(run["predicted"], run["ground_truth"])
        print(
            f"{model:<20} "
            f"{run['receipt']:<25} "
            f"{run['time']:>6.2f}s "
            f"{scores['avg_item_name_score']:>10.2f} "
            f"{scores['item_price_accuracy']:>9.1%} "
            f"{scores['avg_charge_name_score']:>9.2f} "
            f"{scores['charge_amount_accuracy']:>8.1%}"
        )
    print()

Model                Receipt                     Time   ItemName  ItemPrice   ChgName    ChgAmt
-----------------------------------------------------------------------------------------------
qwen2.5vl:3b         bon-solaria.jpeg           13.08s       0.96     80.0%      0.50    50.0%
qwen2.5vl:3b         bon-paul-bakery.jpg         5.59s       0.97    100.0%      1.00   100.0%
qwen2.5vl:3b         bon-jco.jpg                14.38s       0.80    100.0%      1.00   100.0%
qwen2.5vl:3b         bon-hokben.jpg              4.93s       0.69      0.0%      0.63    50.0%

gemma4:e2b           bon-solaria.jpeg           24.69s       0.96    100.0%      0.50     0.0%
gemma4:e2b           bon-paul-bakery.jpg        14.06s       0.67     33.3%      0.23     0.0%
gemma4:e2b           bon-jco.jpg                12.55s       0.29      0.0%      1.00   100.0%
gemma4:e2b           bon-hokben.jpg             13.84s       0.93    100.0%      0.00     0.0%

